# Your Online Dating Advisor Chatbot


In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
load_dotenv()
from llama_index.llms.groq import Groq
from llama_index.core.node_parser import SentenceSplitter
import requests
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.prompts import PromptTemplate
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.base.llms.types import ChatMessage, MessageRole
import gradio as gr
from PIL import Image, ImageDraw                                        # pillow allows drawing the images for the different user/chatbot

### Get the user data

In [3]:
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN")

### Setting up the LLM 

In [4]:
model = "openai/gpt-oss-120b"

llm = Groq(
    model=model,
    api_key=os.environ.get(
        "GROQ_API_KEY"
    ),  
)

### Store the needed data on the topic

In [5]:
os.makedirs("content/data", exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

files = {
    "dating_basics.txt": "https://www.plannedparenthood.org/learn/relationships/healthy-relationships",
    "dating_safety.txt": "https://consumer.ftc.gov/node/298379",
    "romance_scams.txt": "https://consumer.ftc.gov/articles/what-know-about-romance-scams",
    "positive_online_dating.txt": "https://www.pewresearch.org/internet/2020/02/06/users-of-online-dating-platforms-experience-both-positive-and-negative-aspects-of-courtship-on-the-web/",
    "dating_statistics.txt": "https://www.pewresearch.org/internet/2023/02/02/from-looking-for-love-to-swiping-the-field-online-dating-in-the-u-s/",
    "scam_case_studies.txt": "https://www.ic3.gov/PSA/2019/PSA190805",
    "healthy_communication.txt": "https://extension.usu.edu/relationships/faq/communication-that-works",
    "coping_with_breakups.txt": "https://www.psychologytoday.com/gb/blog/the-high-functioning-hotspot/202008/the-ultimate-guide-breakups",
    "harassment_and_boundaries.txt": "https://www.pewresearch.org/internet/2023/02/02/the-experiences-of-u-s-online-daters/",
    "positive_online_dating.txt": "https://www.pewresearch.org/internet/2020/02/06/users-of-online-dating-platforms-experience-both-positive-and-negative-aspects-of-courtship-on-the-web/",
    "negative_online_dating.txt": "https://www.pewresearch.org/internet/2023/02/02/the-experiences-of-u-s-online-daters/",
}

for filename, url in files.items():
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        with open(f"content/data/{filename}", "wb") as f:
            f.write(response.content)
        print(f"Saved {filename}")
    except requests.RequestException as e:
        print(f"Failed to fetch {filename}: {e}")

Saved dating_basics.txt
Saved dating_safety.txt
Saved romance_scams.txt
Saved positive_online_dating.txt
Saved dating_statistics.txt
Saved scam_case_studies.txt
Saved healthy_communication.txt
Saved coping_with_breakups.txt
Saved harassment_and_boundaries.txt
Saved negative_online_dating.txt


### Load the data
LlamaIndex has [loaders](https://docs.llamaindex.ai/en/stable/module_guides/loading/)



In [8]:
documents = SimpleDirectoryReader("content/data").load_data()

In [9]:
documents[0].metadata

{'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\coping_with_breakups.txt',
 'file_name': 'coping_with_breakups.txt',
 'file_type': 'text/plain',
 'file_size': 131985,
 'creation_date': '2026-09-24',
 'last_modified_date': '2026-09-28'}

### Split the document


In [11]:
text_splitter = SentenceSplitter(chunk_size=800, chunk_overlap=150)
docs = text_splitter.get_nodes_from_documents(documents)

In [13]:
docs[0]

TextNode(id_='d8085853-6f4f-4997-8b5f-182d2bd59b80', embedding=None, metadata={'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\coping_with_breakups.txt', 'file_name': 'coping_with_breakups.txt', 'file_type': 'text/plain', 'file_size': 131985, 'creation_date': '2026-09-24', 'last_modified_date': '2026-09-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ed103a26-0448-4481-a0cb-63b407f4c714', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\coping_with_breakups.txt', 'file_name': 'coping_with_breakups.txt', 'file_type': 'text/plain', 'file_size': 131985, 'creation_date': '2026-09-24', 'last_modified_date':

### Create vectors with embeddings

In [13]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings_folder = "/content/embedding_model/"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

2026-09-28 10:50:40,437 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2026-09-28 10:50:42,095 - INFO - 2 prompts are loaded, with the keys: ['query', 'text']


In [15]:
# example of using embeddings to transform sentences into a vector
test_text = "Is it safe to go alone on a first date with someone you met online?"
query_result = embeddings.get_text_embedding(test_text)
# query_result

In [16]:
characters = len(test_text)
dimensions = len(query_result)
print(
    f"The {characters} character sentence was transformed into a {dimensions} dimension vector"
)

The 67 character sentence was transformed into a 384 dimension vector


### Create a vector database

In [18]:
# attempts to read all documents in a folder as text
documents = SimpleDirectoryReader("content/data").load_data()

vector_index = VectorStoreIndex.from_documents(
    documents, transformations=[text_splitter], embed_model=embeddings
)

# save it to use over and over
vector_index.storage_context.persist(persist_dir="/content/vector_index")

### Add a prompt

In [ ]:
input_template = """Here is the context:
{context_str}

Answer the question based only on the following context. Keep your answers short and succinct.
Question to be answered: {query_str}
Answer:"""

prompt = PromptTemplate(template=input_template)

### RAG - chain it all together

In [20]:
query_engine = vector_index.as_query_engine(
    llm=llm, text_qa_template=prompt, similarity_top_k=2, response_mode="tree_summarize"
)

In [21]:
#example
answer = query_engine.query("Do people end up meeting their forever partners online?")
print(answer)

2026-09-28 10:54:19,070 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Yes—some people do find lasting, even lifelong, partners through online dating. Surveys show that a portion of users report meeting a partner on a dating site or app, though the majority use these platforms for a range of relationship goals and experiences vary widely.


#### We can explore the answer

In [22]:
answer.__dict__.keys()
answer.metadata

{'5c24b1c3-c449-4c4d-80ca-804eb65bca16': {'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\dating_statistics.txt',
  'file_name': 'dating_statistics.txt',
  'file_type': 'text/plain',
  'file_size': 460666,
  'creation_date': '2026-09-24',
  'last_modified_date': '2026-09-28'},
 '9bff17c6-61ea-42e6-a8d2-ad685632e2e1': {'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\positive_online_dating.txt',
  'file_name': 'positive_online_dating.txt',
  'file_type': 'text/plain',
  'file_size': 441498,
  'creation_date': '2026-09-24',
  'last_modified_date': '2026-09-28'}}

##### `response`

The response.

In [25]:
answer.response
print(answer.response)

Yes—some people do find lasting, even lifelong, partners through online dating. Surveys show that a portion of users report meeting a partner on a dating site or app, though the majority use these platforms for a range of relationship goals and experiences vary widely.


##### `source_nodes`

In [ ]:
answer.source_nodes
answer.source_nodes[0]
answer.source_nodes[-1].text
print(answer.source_nodes[-1].text)

[NodeWithScore(node=TextNode(id_='f2ef52d8-79ed-48c0-aaa0-bc819d756027', embedding=None, metadata={'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\dating_statistics.txt', 'file_name': 'dating_statistics.txt', 'file_type': 'text/plain', 'file_size': 460646, 'creation_date': '2026-09-24', 'last_modified_date': '2026-09-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='6e423507-bf7c-49f1-8b7d-242f19b8d98a', node_type='4', metadata={'file_path': 'c:\\Users\\berth\\OneDrive\\Documents\\GenAI\\content\\data\\dating_statistics.txt', 'file_name': 'dating_statistics.txt', 'file_type': 'text/plain', 'file_size': 460646, 'creation_date': '2026-09-24', 'last_modified_date': '2026-09-25'},

##### `metadata`

In [26]:
answer.metadata
list(answer.metadata.values())[0]
list(answer.metadata.values())[0]["file_name"]

'dating_statistics.txt'

## RAG with memory

In [28]:
model = "openai/gpt-oss-120b"

llm = Groq(
    model=model,
    api_key=os.environ.get("GROQ_API_KEY"),
)

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings_folder = "/content/embedding_model/"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model, cache_folder=embeddings_folder
)

storage_context = StorageContext.from_defaults(persist_dir="/content/vector_index")
vector_index = load_index_from_storage(storage_context, embed_model=embeddings)
retriever = vector_index.as_retriever(similarity_top_k=2)

prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "You are DatingCoach, a warm and knowledgeable advisor who helps people "
            "navigate online and offline dating. You speak like a supportive, "
            "experienced friend — encouraging but honest, never judgmental."
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "Answer using only the provided context and the prior conversation. "
            "If the context doesn't cover the question, do not say things like "
            "'the information provided does not include that' or refer to "
            "'the context' or 'the documents' at all. Instead, say something like "
            "'As far as my knowledge goes, I'm not able to help with that' or "
            "'That's outside what I currently know — I'd rather not guess.'"
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "Prioritize the user's safety and wellbeing above all else. When relevant, "
            "surface safety practices (meeting in public, video-verifying identity, "
            "not sharing financial or personal info early) and be alert to red flags "
            "like romance-scam patterns, pressure, or boundary violations."
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "Keep answers concise (2-4 sentences unless the user asks for more detail), "
            "practical, and specific — prefer concrete next steps over generic platitudes."
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "You are not a therapist or law enforcement. For situations involving abuse, "
            "harassment, stalking, or immediate danger, acknowledge the seriousness and "
            "point the person toward appropriate professional or emergency resources "
            "rather than trying to resolve it yourself."
        ),
    ),
]

memory = ChatMemoryBuffer.from_defaults()

rag_bot = ContextChatEngine(
    llm=llm, retriever=retriever, memory=memory, prefix_messages=prefix_messages
)

2026-09-28 10:58:26,914 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Eine vorhandene Verbindung wurde vom Remotehost geschlossen', None, 10054, None)), '(Request ID: 7d031070-6da8-4ea0-a768-df036c2096ab)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
2026-09-28 10:58:26,928 - WARNING - '(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'Eine vorhandene Verbindung wurde vom Remotehost geschlossen', None, 10054, None)), '(Request ID: 7d031070-6da8-4ea0-a768-df036c2096ab)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
2026-09-28 10:58:26,929 - WARNING - Retrying in 1s [Retry 1/5].
2026-09-28 10:58:29,448 - INFO - 2 prompts are loaded, with the keys: ['query', 'text']
2026-09-28 10:58:33,497 - INF

In [ ]:
# example
ans = rag_bot.chat("Why are some people scared of online dating?")
print(ans.response)

2026-09-25 15:29:34,746 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Many people worry that online dating isn’t safe—especially women and older adults—because they hear stories of fake profiles, harassment, or scams, and they notice that those who haven’t used a dating app feel less secure than seasoned users. These safety fears are reinforced by the fact that a solid majority (about 60 percent) think platforms should require background checks before anyone can create a profile. If you’re feeling uneasy, start by using apps that offer identity verification, keep conversations on the platform, and always meet in a public place until you’re comfortable. Taking those steps can help turn anxiety into confidence.


## Gradio - Creating the App

In [ ]:
# Generate avatar images 
os.makedirs("avatars", exist_ok=True)

def make_avatar(path, bg_color, draw_fn, size=256):
    img = Image.new("RGB", (size, size), bg_color)
    draw = ImageDraw.Draw(img)
    draw_fn(draw)
    mask = Image.new("L", (size, size), 0)
    ImageDraw.Draw(mask).ellipse((0, 0, size, size), fill=255)
    img.putalpha(mask)
    img.save(path)

def draw_heart(d):
    d.ellipse((52, 60, 140, 148), fill="white")
    d.ellipse((116, 60, 204, 148), fill="white")
    d.polygon([(54, 112), (202, 112), (128, 206)], fill="white")

def draw_person(d):
    d.ellipse((98, 56, 158, 116), fill="white")                         # head
    d.ellipse((66, 132, 190, 268), fill="white")                        # shoulders

make_avatar("avatars/bot.png", "#e11d48", draw_heart)                 # saving in a folder
make_avatar("avatars/user.png", "#d9a25f", draw_person)

# define theme of chat boxes
theme = gr.themes.Soft(
    primary_hue="rose",
    secondary_hue="amber",
    neutral_hue="stone",
    text_size=gr.themes.sizes.text_lg,
    font=[gr.themes.GoogleFont("Lora"), "ui-serif", "serif"],           # these are found here https://fonts.google.com
)

# Custom Cascading Style Sheets
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #fff5f7 0%, #fff9ed 100%);   
}
#component-0 {
    max-width: 820px;
    margin: auto;
}
.message.user {
    background: linear-gradient(135deg, #f8d7e3, #fbe4c5) !important;
    border-radius: 18px !important;
    color: #3b2a30 !important;
}
.message.user, .message.bot {
    font-size: 20px !important;
}
h1 {
    font-size: 2.4rem !important;
    text-align: center;
    font-weight: 600;
}
.gradio-container p {
    font-size: 1.15rem !important;
    text-align: center;
}
.message.bot {
    background: #ffffff !important;
    border: 1px solid #f0c9d6 !important;
    border-radius: 18px !important;
    color: #3b2a30 !important;
    box-shadow: 0 2px 8px rgba(225, 29, 72, 0.06);
}
h1 {
    text-align: center;
    font-weight: 600;
}
.gradio-container p {
    text-align: center;
}
"""

# Chat callback 
def rag_bot_chat_interface(message, history):
    response = rag_bot.chat(message)
    return response.response

# Interface
demo_chat = gr.ChatInterface(
    fn=rag_bot_chat_interface,
    title="Your Online Dating Advisor",
    description="Discreet, evidence-based advice on dating safely, spotting red flags, and navigating modern romance.",
    examples=[
        "How do I know if someone's a romance scammer?",
        "What should one look for as warning signs on a first date?",
        "Which online dating apps are highly rated?",
    ],
    chatbot=gr.Chatbot(
        avatar_images=("avatars/user.png", "avatars/bot.png"),  # (user, chatbot)
        show_label=False,
        height=520,
    ),
    textbox=gr.Textbox(placeholder="Ask your advisor anything...", container=False, scale=7),
)

demo_chat.launch(theme=theme, share=True, css=custom_css)

2026-09-28 11:01:25,273 - INFO - HTTP Request: GET http://127.0.0.1:7874/gradio_api/startup-events "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7874


2026-09-28 11:01:25,499 - INFO - HTTP Request: HEAD http://127.0.0.1:7874/ "HTTP/1.1 200 OK"
2026-09-28 11:01:25,919 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-09-28 11:01:26,289 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


* Running on public URL: https://5fba0ab2af1267ccc5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-09-28 11:01:29,619 - INFO - HTTP Request: HEAD https://5fba0ab2af1267ccc5.gradio.live "HTTP/1.1 200 OK"
